# Using Objective Adapters

In **crystALL**, property predictions (objectives) are generated using inference models that process a structural object. These models include static models, which do not alter the input structure, as well as Machine Learning Interatomic Potentials (MLIPs), which update the structure by relaxing it. To manage a wide range of model backends from the external community, **crystALL** utilizes calculator **Adapters**.

This tutorial covers:
1. Running adapters via the high-level `CrystALLSearch.get_calculator_objectives` API.
2. Defining custom objectives as combinations of existing predictions.
3. Importing and using adapters directly from the `calculators` module.
4. Using **Piped Calculations** to chain relaxation and static property prediction.

### Initial Setup

We start by identifying a few prototypes for `BaSO4` to work with, similar to Tutorial 1.

In [ ]:
%%capture
from crystall import CrystALLSearch
from pymongo import MongoClient
import os

# Initialize and generate a few prototypes
crystallSearch = CrystALLSearch("BaSO4")
client_string = os.getenv('MONGODB_URI')
if client_string:
    MPDD_database = MongoClient(client_string)['MPDD']
    crystallSearch.setup_MPDD(MPDD_database)
    # Generate a very small set for this tutorial
    crystallSearch.generate_MPDD(limit=5)
else:
    print("MONGODB_URI not found. Please ensure you have access to MPDD or provide local structures.")

### Using `get_calculator_properties`

The easiest way to use adapters is through the `CrystALLSearch` instance. This method automatically discovers installed adapters, initializes their micro-environments (using `uv`), and merges the results back into the search state.

In [ ]:
# Run multiple MLIP predictions through the high-level API
crystallSearch.get_calculator_objectives(
    objective_codes=["MACE_RELAX_E", "CHGNET_RELAX_E"], 
    device="cpu"
)

### Defining Custom Objectives

You can define new objectives as arithmetic combinations of existing ones using the `define_objective` method. This is useful for averaging predictions from multiple models, calculating energy differences, or scaling values.

The `definition` string uses `#0`, `#1`, etc., to reference the indices in the `objectives` list.

In [ ]:
# Define a new objective as the average of MACE and CHGNet predictions
crystallSearch.define_objective(
    name="Avg_MLIP_Energy",
    objectives=["MACE_RELAX_E", "CHGNET_RELAX_E"],
    definition="(#0 + #1) / 2"
)

# Define another objective as a simple difference
crystallSearch.define_objective(
    name="MLIP_Disagreement",
    objectives=["MACE_RELAX_E", "CHGNET_RELAX_E"],
    definition="abs(#0 - #1)"
)

In [ ]:
import pandas as pd
df = pd.DataFrame(crystallSearch.all_prototypes)
cols = ["name", "MACE_RELAX_E", "CHGNET_RELAX_E", "Avg_MLIP_Energy", "MLIP_Disagreement"]

if all(c in df.columns for c in cols):
    display(df[cols].head())
else:
    print("One or more objective columns missing. Ensure calculations and definitions succeeded.")

### Using Adapters Directly

Sometimes you may want to use an adapter outside of the `CrystALLSearch` workflow, for example, in a custom script or to test a specific model configuration. All adapters are re-exported under `crystall.calculators`.

In [ ]:
from crystall.calculators import MLIPAdapter, MatGLAdapter

# Initialize a MACE adapter directly
mace = MLIPAdapter("MACE", device="cpu")

# One-time setup of the micro-environment
mace.setup_env()

Once initialized, you can run the adapter on a single structure file or an entire directory. Let's run it on one of our generated prototypes.

In [ ]:
prototype_dir = "Ba1S1O4-MPDD(curated)-prototypes"
if os.path.exists(prototype_dir):
    # List all files in the directory
    prototype_files = sorted([f for f in os.listdir(prototype_dir) if f.endswith('.poscar') or f.endswith('.cif')])
    if prototype_files:
        first_prototype = os.path.join(prototype_dir, prototype_files[0])
        
        # Run on a single structure
        result = mace.run_on_structure(first_prototype)
        print(f"--- Single Structure Result ({first_prototype}) ---")
        print(result['objectives'])
    else:
        print(f"No prototype files found in {prototype_dir}")
else:
    print(f"Directory {prototype_dir} not found.")

### Batch Processing with `run_on_directory`

For large sets of structures, `run_on_directory` is significantly more efficient as it handles internal batching and minimizes micro-environment startup overhead.

In [ ]:
if os.path.exists(prototype_dir):
    # Run on the entire directory
    dir_results = mace.run_on_directory(prototype_dir, batch=5)
    
    print(f"--- Directory Results (Processed {len(dir_results)} structures) ---")
    # Display the first few results
    for res in dir_results[:2]:
        print(f"{res['filename']}: {res['objectives']}")
else:
    print(f"Directory {prototype_dir} not found.")

### Piped Calculations

**MLIPAdapter** features a powerful "piping" capability. It can perform a full structural relaxation using an MLIP (like MACE or CHGNet) and then automatically pipe the relaxed structure to one or more **static** adapters (like MatGL or ALIGNN) to calculate additional properties (e.g., formation energy, bandgap) on the optimized geometry.

In [ ]:
# Initialize a static adapter
matgl = MatGLAdapter(device="cpu")

# Initialize MLIPAdapter with the static adapter in its pipe
mace_piped = MLIPAdapter("MACE", device="cpu", static=[matgl])

# Ensure both environments are ready
mace_piped.setup_env()

# Run the piped calculation
if os.path.exists(prototype_dir):
    piped_result = mace_piped.run_on_structure(first_prototype)

    print("Piped Result (Notice the namespaced keys):")
    for key in piped_result['objectives']:
        if "MATGL" in key:
            print(f"{key}: {piped_result['objectives'][key]}")
else:
    print(f"Directory {prototype_dir} not found.")

In the results above, keys starting with `MACE_MATGL_` represent properties calculated by MatGL *after* MACE relaxed the structure.

### Summary and Best Practices

- **High-level API**: Use `crystallSearch.get_calculator_properties` for most research tasks. It keeps everything integrated.
- **Custom Objectives**: Use `define_objective` to create complex criteria (e.g., multi-model consensus or property ratios) for filtering candidates.
- **Direct Adapters**: Use `MLIPAdapter`, `MatGLAdapter`, etc., for custom scripting or isolated testing.
- **Piping**: Always pipe static properties into relaxation steps if you want properties calculated on the equilibrium geometry.
- **Device Management**: Use `device="cpu"` for testing and `device="cuda"` for large-scale GPU acceleration.